In [ ]:
import pandas as pd

df1 = pd.read_csv("./data/output_extract_llm_1909_01_22.csv")
df2 = pd.read_csv("./data/output_extract_llm_2.csv")
df3 = pd.read_csv("./data/output_extract_llm_complet.csv")
df4 = pd.read_csv("./data/output_extract_llm.csv")


In [3]:
df["date"].unique()

array(['1907-11-26', '1907-11-27', '1908-11-20', '1908-11-21',
       '1908-11-26', '1909-11-18'], dtype=object)

In [2]:
df_total = pd.concat([df1, df2, df3,df4], ignore_index=True)

In [3]:
df_total["date"].unique()

array(['1909-01-22', '1907-06-28', '1908-12-11', '1909-12-07',
       '1907-11-26', '1907-11-27', '1908-11-20', '1908-11-21',
       '1908-11-26', '1909-11-18', '1881-01-11', '1881-01-20'],
      dtype=object)

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

texts = df_total["prise_de_parole"].tolist()

embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

topic_model = BERTopic(embedding_model=embedding_model, language="multilingual", min_topic_size=20)

topics, probabilities = topic_model.fit_transform(texts)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
df_noms=(df_total.groupby("nom", as_index=False).agg({"prise_de_parole": "\n\n".join}))

In [ ]:
import pandas as pd

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer


print("Dimensions :", df_noms.shape)
print("Colonnes :", df_noms.columns.tolist())

colonnes_necessaires = ["nom","prise_de_parole"]

for colonne in colonnes_necessaires:
    if colonne not in df_noms.columns:
        raise ValueError(f"La colonne '{colonne}' n'existe pas dans df_noms")

df_noms["prise_de_parole"] = (df_noms["prise_de_parole"].fillna("").astype(str).str.strip())
df_noms = df_noms[df_noms["prise_de_parole"].str.len() > 0].copy()
df_noms = df_noms.reset_index(drop=True)

texts = df_noms["prise_de_parole"].tolist()
print("Nombre de prises de parole :", len(texts))


embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
stop_words_fr = liste_mots
vectorizer_model = CountVectorizer(stop_words=stop_words_fr)


topic_model = BERTopic(embedding_model=embedding_model,vectorizer_model=vectorizer_model,language="multilingual",min_topic_size=20,verbose=True)

topics, probabilities = topic_model.fit_transform(texts)


if len(topics) != len(df_noms):
    raise ValueError(
        f"Nombre de topics ({len(topics)}) différent du nombre de lignes ({len(df_noms)})")
df_noms["topic"] = topics

if probabilities is not None:

    try:
        df_noms["probabilite_topic"] = [max(p) if hasattr(p, "__len__") else p for p in probabilities]

    except Exception:
        print("Impossible de calculer la probabilité du topic.")

def mots_du_topic(topic, nombre_mots=10):
    if topic == -1:
        return []

    mots = topic_model.get_topic(topic)

    if not mots:
        return []

    return [mot for mot, score in mots[:nombre_mots]]


df_noms["mots_topic"] = df_noms["topic"].apply(mots_du_topic)


print("\nPremières lignes :")

print(df_noms[["nom","topic","mots_topic","prise_de_parole"]].head())


topic_info = topic_model.get_topic_info()
print("\n==============================")
print("TOPICS")
print("==============================")
print(topic_info)
print("\n==============================")
print("MOTS DES TOPICS")
print("==============================")

for topic in topic_model.get_topics():

    if topic == -1:
        continue

    mots = topic_model.get_topic(topic)

    print(f"\nTOPIC {topic}")

    for mot, score in mots[:10]:
        print(f"  {mot:25} {score:.4f}")


df_noms.to_pickle("df_noms_topics.pkl")

print("\nDataFrame sauvegardé : df_noms_topics.pkl")

Dimensions : (2156, 5)
Colonnes : ['nom', 'prise_de_parole', 'topic', 'probabilite_topic', 'mots_topic']
Nombre de prises de parole : 2156


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-08-23 15:53:37,663 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/68 [00:00<?, ?it/s]

2026-08-23 15:55:27,839 - BERTopic - Embedding - Completed ✓
2026-08-23 15:55:27,844 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-23 15:55:51,140 - BERTopic - Dimensionality - Completed ✓
2026-08-23 15:55:51,143 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-23 15:55:51,396 - BERTopic - Cluster - Completed ✓
2026-08-23 15:55:51,406 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-23 15:55:52,137 - BERTopic - Representation - Completed ✓



Premières lignes :
                                                 nom  topic  \
0                                          !hot.lees      0   
1                                                  '      0   
2                                         (Ardennes)      0   
3  (Ce chiffre, mis aux voix, est adopté.) M. le ...      0   
4                                           (ECclhet      1   

                                          mots_topic  \
0  [ministre, faire, loi, chambre, compagnie, tra...   
1  [ministre, faire, loi, chambre, compagnie, tra...   
2  [ministre, faire, loi, chambre, compagnie, tra...   
3  [ministre, faire, loi, chambre, compagnie, tra...   
4  [bao, lramel, lie, persuasion, constant, ain, ...   

                                     prise_de_parole  
0  Et ils ajoutent : « L'Autriche restitue à la T...  
1                                           Cornand.  
2                    JacquesDufour\n\nJacques Dufour  
3  En conséquence, le chapitre 59 est adopté

In [26]:
with open("/home/port-pret-etu01/Documents/memoire_git/stop_words_fr","r",encoding="utf-8") as liste :
    mots=liste.read().split("\\")

mots

["a\nabord\nabsolument\nafin\nah\nai\naie\naient\naies\nailleurs\nainsi\nait\nallaient\nallo\nallons\nallô\nalors\nanterieur\nanterieure\nanterieures\napres\naprès\nas\nassez\nattendu\nau\naucun\naucune\naucuns\naujourd\naujourd'hui\naupres\nauquel\naura\naurai\nauraient\naurais\naurait\nauras\naurez\nauriez\naurions\naurons\nauront\naussi\nautant\nautre\nautrefois\nautrement\nautres\nautrui\naux\nauxquelles\nauxquels\navaient\navais\navait\navant\navec\navez\naviez\navions\navoir\navons\nayant\nayez\nayons\nb\nbah\nbas\nbasee\nbat\nbeau\nbeaucoup\nbien\nbigre\nbon\nboum\nbravo\nbrrr\nc\ncar\nce\nceci\ncela\ncelle\ncelle-ci\ncelle-là\ncelles\ncelles-ci\ncelles-là\ncelui\ncelui-ci\ncelui-là\ncelà\ncent\ncependant\ncertain\ncertaine\ncertaines\ncertains\ncertes\nces\ncet\ncette\nceux\nceux-ci\nceux-là\nchacun\nchacune\nchaque\ncher\nchers\nchez\nchiche\nchut\nchère\nchères\nci\ncinq\ncinquantaine\ncinquante\ncinquantième\ncinquième\nclac\nclic\ncombien\ncomme\ncomment\ncomparable\ncompar

In [29]:
liste_mots=[]
for i in range(len(mots)):
    for m in(mots[i].split()):
        liste_mots.append(m)

In [30]:
liste_mots

['a',
 'abord',
 'absolument',
 'afin',
 'ah',
 'ai',
 'aie',
 'aient',
 'aies',
 'ailleurs',
 'ainsi',
 'ait',
 'allaient',
 'allo',
 'allons',
 'allô',
 'alors',
 'anterieur',
 'anterieure',
 'anterieures',
 'apres',
 'après',
 'as',
 'assez',
 'attendu',
 'au',
 'aucun',
 'aucune',
 'aucuns',
 'aujourd',
 "aujourd'hui",
 'aupres',
 'auquel',
 'aura',
 'aurai',
 'auraient',
 'aurais',
 'aurait',
 'auras',
 'aurez',
 'auriez',
 'aurions',
 'aurons',
 'auront',
 'aussi',
 'autant',
 'autre',
 'autrefois',
 'autrement',
 'autres',
 'autrui',
 'aux',
 'auxquelles',
 'auxquels',
 'avaient',
 'avais',
 'avait',
 'avant',
 'avec',
 'avez',
 'aviez',
 'avions',
 'avoir',
 'avons',
 'ayant',
 'ayez',
 'ayons',
 'b',
 'bah',
 'bas',
 'basee',
 'bat',
 'beau',
 'beaucoup',
 'bien',
 'bigre',
 'bon',
 'boum',
 'bravo',
 'brrr',
 'c',
 'car',
 'ce',
 'ceci',
 'cela',
 'celle',
 'celle-ci',
 'celle-là',
 'celles',
 'celles-ci',
 'celles-là',
 'celui',
 'celui-ci',
 'celui-là',
 'celà',
 'cent',
 '

KeyError: 'mots_topic'

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import itertools
import json

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Dimensions :", df_noms.shape)
print("Colonnes :", df_noms.columns.tolist())

colonnes_necessaires = ["nom", "prise_de_parole"]

for colonne in colonnes_necessaires:
    if colonne not in df_noms.columns:
        raise ValueError(f"La colonne '{colonne}' n'existe pas dans df_noms")


df_noms["prise_de_parole"] = (df_noms["prise_de_parole"].fillna("").astype(str).str.strip())

df_noms = df_noms[df_noms["prise_de_parole"].str.len() > 0].copy()
df_noms = df_noms.reset_index(drop=True)

print("Nombre de prises de parole :", len(df_noms))
print("Nombre de personnes distinctes :", df_noms["nom"].nunique())


embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")


texts = df_noms["prise_de_parole"].tolist()

print("Calcul des embeddings (peut prendre un moment)...")
embeddings = embedding_model.encode(texts,show_progress_bar=True,batch_size=32)
df_noms["embedding"] = list(embeddings)

personnes = df_noms["nom"].unique().tolist()

vecteurs_personnes = {}
nb_prises_par_personne = {}

for personne in personnes:
    sous_df = df_noms[df_noms["nom"] == personne]
    vecs = np.stack(sous_df["embedding"].values)
    vecteur_moyen = vecs.mean(axis=0)
    vecteurs_personnes[personne] = vecteur_moyen
    nb_prises_par_personne[personne] = len(sous_df)

print("\nNombre de prises de parole par personne :")
for p, n in sorted(nb_prises_par_personne.items(), key=lambda x: -x[1]):
    print(f"  {p:30} {n}")


noms_list = list(vecteurs_personnes.keys())
matrice_vecteurs = np.stack([vecteurs_personnes[n] for n in noms_list])

sim_matrix = cosine_similarity(matrice_vecteurs)

sim_df = pd.DataFrame(sim_matrix, index=noms_list, columns=noms_list)

print("\nMatrice de similarité (extrait) :")
print(sim_df.round(3))

MIN_SIMILARITY = 0.5

G = nx.Graph()

for nom in noms_list:
    G.add_node(nom,nb_prises_de_parole=nb_prises_par_personne[nom])

for nom_a, nom_b in itertools.combinations(noms_list, 2):
    sim = sim_matrix[noms_list.index(nom_a), noms_list.index(nom_b)]
    if sim >= MIN_SIMILARITY:
        G.add_edge(nom_a, nom_b, weight=round(float(sim), 4))

print(f"\nGraphe : {G.number_of_nodes()} nœuds, {G.number_of_edges()} arêtes")


paires_sim = []
for nom_a, nom_b in itertools.combinations(noms_list, 2):
    sim = sim_matrix[noms_list.index(nom_a), noms_list.index(nom_b)]
    paires_sim.append((nom_a, nom_b, sim))

paires_df = pd.DataFrame(paires_sim, columns=["personne_a", "personne_b", "similarite"])

print("\nDistribution des similarités :")
print(paires_df["similarite"].describe())

print("\nTop 20 paires les plus similaires :")
print(paires_df.sort_values("similarite", ascending=False).head(20))


nx.write_gexf(G, "graphe_individus_semantique.gexf")
nx.write_graphml(G, "graphe_individus_semantique.graphml")

sim_df.to_csv("matrice_similarite_personnes.csv", encoding="utf-8")
paires_df.sort_values("similarite", ascending=False).to_csv("paires_similarite_personnes.csv", index=False, encoding="utf-8")

top_edges = sorted(G.edges(data=True), key=lambda e: -e[2]["weight"])[:20]
with open("top_relations_semantiques.json", "w", encoding="utf-8") as f:
    json.dump([{"personne_a": a, "personne_b": b, "similarite": d["weight"]} for a, b, d in top_edges],f, ensure_ascii=False, indent=2,)

print("\n[✓] Terminé. Fichiers écrits :")
print("  - graphe_individus_semantique.gexf (ouvrable dans Gephi)")
print("  - graphe_individus_semantique.graphml")
print("  - matrice_similarite_personnes.csv")
print("  - paires_similarite_personnes.csv")
print("  - top_relations_semantiques.json")

Dimensions : (2156, 5)
Colonnes : ['nom', 'prise_de_parole', 'topic', 'probabilite_topic', 'mots_topic']
Nombre de prises de parole : 2156
Nombre de personnes distinctes : 2156


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Calcul des embeddings (peut prendre un moment)...


Batches:   0%|          | 0/68 [00:00<?, ?it/s]


Nombre de prises de parole par personne :
  !hot.lees                      1
  '                              1
  (Ardennes)                     1
  (Ce chiffre, mis aux voix, est adopté.) M. le président 1
  (ECclhet                       1
  (Exclama tions ironiques)      1
  (Gabriel)                      1
  (La commission)                1
  (Les partisans de la Faucille) 1
  (Mouvement divers)             1
  (Mouvements divers)            1
  (On rit)                       1
  (PBy.Bellier-Benazet           1
  (Très bien! très bien!)        1
  (Très bienl très bien 1)       1
  *mins                          1
  , Charles Benoist.             1
  , Schneider (Charles) (Haut-Rhin) 1
  -                              1
  - Rougier                      1
  -+'■Hubert (Lucien*            1
  -.-                            1
  -Ouderc.                       1
  -Pres-sensé                    1
  -lJ.UClle-=.vJ.                1
  .p                             1
  0, Moncher collèg

KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import itertools
import json

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity


print("Dimensions :", df_noms.shape)
print("Colonnes :", df_noms.columns.tolist())

colonnes_necessaires = ["nom", "prise_de_parole"]

for colonne in colonnes_necessaires:
    if colonne not in df_noms.columns:
        raise ValueError(f"La colonne '{colonne}' n'existe pas dans df_noms")


df_noms["prise_de_parole"] = (df_noms["prise_de_parole"].fillna("").astype(str).str.strip())

df_noms = df_noms[df_noms["prise_de_parole"].str.len() > 0].copy()
df_noms = df_noms.reset_index(drop=True)

print("Nombre de prises de parole :", len(df_noms))
print("Nombre de personnes distinctes :", df_noms["nom"].nunique())


embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

texts = df_noms["prise_de_parole"].tolist()

print("Calcul des embeddings (peut prendre un moment)...")
embeddings = embedding_model.encode(texts,show_progress_bar=True,batch_size=32)

NB_THEMES = 15

kmeans = KMeans(n_clusters=NB_THEMES,random_state=42,n_init=10)

themes = kmeans.fit_predict(embeddings)
df_noms["theme"] = themes

print("\nRépartition des prises de parole par thème :")
print(df_noms["theme"].value_counts().sort_index())

def exemples_du_theme(theme_id, nb_exemples=3):
    sous_df = df_noms[df_noms["theme"] == theme_id]
    idx_sous_df = sous_df.index.tolist()
    centre = kmeans.cluster_centers_[theme_id]
    distances = [(idx, np.linalg.norm(embeddings[idx] - centre)) for idx in idx_sous_df]
    distances.sort(key=lambda x: x[1])
    return [df_noms.loc[idx, "prise_de_parole"][:150] for idx, _ in distances[:nb_exemples]]


print("\n==============================")
print("APERÇU DES THÈMES")
print("==============================")
for theme_id in range(NB_THEMES):
    print(f"\nTHÈME {theme_id} ({(df_noms['theme'] == theme_id).sum()} prises de parole)")
    for ex in exemples_du_theme(theme_id):
        print(f"  - {ex}")

personnes = df_noms["nom"].unique().tolist()

vecteurs_personnes = {}
nb_prises_par_personne = {}

for personne in personnes:
    sous_df = df_noms[df_noms["nom"] == personne]
    nb_prises_par_personne[personne] = len(sous_df)

    compte_par_theme = sous_df["theme"].value_counts()
    vecteur = np.zeros(NB_THEMES)
    for theme_id, count in compte_par_theme.items():
        vecteur[theme_id] = count

    if vecteur.sum() > 0:
        vecteur = vecteur / vecteur.sum()

    vecteurs_personnes[personne] = vecteur

print("\nNombre de prises de parole par personne :")
for p, n in sorted(nb_prises_par_personne.items(), key=lambda x: -x[1]):
    print(f"  {p:30} {n}")


noms_list = list(vecteurs_personnes.keys())
matrice_vecteurs = np.stack([vecteurs_personnes[n] for n in noms_list])

sim_matrix = cosine_similarity(matrice_vecteurs)
sim_df = pd.DataFrame(sim_matrix, index=noms_list, columns=noms_list)

print("\nMatrice de similarité (extrait) :")
print(sim_df.round(3))


paires_sim = []
for nom_a, nom_b in itertools.combinations(noms_list, 2):
    sim = sim_matrix[noms_list.index(nom_a), noms_list.index(nom_b)]
    paires_sim.append((nom_a, nom_b, sim))

paires_df = pd.DataFrame(paires_sim, columns=["personne_a", "personne_b", "similarite"])

print("\nDistribution des similarités :")
print(paires_df["similarite"].describe())

print("\nTop 20 paires les plus similaires :")
print(paires_df.sort_values("similarite", ascending=False).head(20))


MIN_SIMILARITY = 0.5

G = nx.Graph()

for nom in noms_list:
    G.add_node(nom,nb_prises_de_parole=nb_prises_par_personne[nom],vecteur_themes_json=json.dumps(vecteurs_personnes[nom].tolist()))

for nom_a, nom_b in itertools.combinations(noms_list, 2):
    sim = sim_matrix[noms_list.index(nom_a), noms_list.index(nom_b)]
    if sim >= MIN_SIMILARITY:
        G.add_edge(nom_a, nom_b, weight=round(float(sim), 4))

print(f"\nGraphe : {G.number_of_nodes()} nœuds, {G.number_of_edges()} arêtes")


nx.write_gexf(G, "graphe_individus_themes.gexf")
nx.write_graphml(G, "graphe_individus_themes.graphml")

sim_df.to_csv("matrice_similarite_themes.csv", encoding="utf-8")
paires_df.sort_values("similarite", ascending=False).to_csv("paires_similarite_themes.csv", index=False, encoding="utf-8")
df_noms.to_pickle("df_noms_themes.pkl")

top_edges = sorted(G.edges(data=True), key=lambda e: -e[2]["weight"])[:20]
with open("top_relations_themes.json", "w", encoding="utf-8") as f:
    json.dump([{"personne_a": a, "personne_b": b, "similarite": d["weight"]} for a, b, d in top_edges],f, ensure_ascii=False, indent=2)

print("\n[✓] Terminé. Fichiers écrits :")
print("  - graphe_individus_themes.gexf (ouvrable dans Gephi)")
print("  - graphe_individus_themes.graphml")
print("  - matrice_similarite_themes.csv")
print("  - paires_similarite_themes.csv")
print("  - top_relations_themes.json")
print("  - df_noms_themes.pkl (dataframe avec la colonne 'theme')")

Dimensions : (2156, 2)
Colonnes : ['nom', 'prise_de_parole']
Nombre de prises de parole : 2156
Nombre de personnes distinctes : 2156


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Calcul des embeddings (peut prendre un moment)...


Batches:   0%|          | 0/68 [00:00<?, ?it/s]


Répartition des prises de parole par thème :
theme
0      46
1     168
2     203
3     172
4     162
5      85
6     304
7     165
8     128
9     201
10     22
11     16
12    242
13    129
14    113
Name: count, dtype: int64

APERÇU DES THÈMES

THÈME 0 (46 prises de parole)
  - Plaie en rigole de l'occiput. Sérieuse.
  - Plaie du foie par balle, décès après
  - Un œdème notable de la lèvre inférîioeureà la partie droite dont la muqueuse àceniveaua été déchirée contre les dents dunaxillaireinférieur. Cette lés

THÈME 1 (168 prises de parole)
  - éduire de 10,000 hectares les surfaces plantées aujourd'hui en vigne à grand rendement, nous aurons diminué la production totale de 1 million d'hectol
  - permettre aux compagnies qui se¡uapent,qui envoient des marchandises.•.uisdemauvaises directions, de dégagerjrrasponsabilité.En voulez-vous des;'it>rn
  - entraver en rien, en effet, des travaux d'amélioration qui doivent contribue; à la richesse des pays traversés par les rivières navigabl

In [ ]:
df_resultats = pd.read_csv("./data/matrice_similarite_themes.csv")

In [39]:
df_resultats

,Unnamed: 0,!hot.lees,',(Ardennes),"(Ce chiffre, mis aux voix, est adopté.) M. le président",(ECclhet,(Exclama tions ironiques),(Gabriel),(La commission),(Les partisans de la Faucille),...,àraisQlens,"àsÂquoiattribuerce laisser-aller,Sectnedelacompagnie",é,éduire,éner'O'iqyraud,éphémère,éptionno,"î^entlaf>^Uce""N?usvenonsde voir comjonctlaCompagmes'estconformée aux mdeVIldduserviceducontrôle au pointPersannatérieletsousle rapport duPersonneCI,Ptrunedi~,,ninutionquo.sechiffreCestparunediminutionque sevezIllilitvuedescombustibles,",ï01auministredes finances,ûe
0,!hot.lees,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,',0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,(Ardennes),0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,"(Ce chiffre, mis aux voix, est adopté.) M. le ...",0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,(ECclhet,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2151,éphémère,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2152,éptionno,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0
2153,"î^entlaf>^Uce""N?usvenonsde voir comjonctlaComp...",0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0
2154,ï01auministredes finances,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0
